**Bloc 1: Configuració del camí i imports**

In [1]:
import sys
sys.path.append("../src")

from domain import (
    Session, User, Category, PaymentMethod, 
    TransactionType, FinancialProfile, Budget
)
from datetime import date
from decimal import Decimal

**Bloc 2: Dades Mestres (Categories, Mètodes i Tipus)**

In [2]:
with Session() as session:
    # 1. CATEGORIES BÀSIQUES
    cat_data = [
        ("Alimentació", "Supermercat i restaurants", True),
        ("Habitatge", "Lloguer, factures i manteniment", True),
        ("Transport", "Benzina, bus i metro", True),
        ("Oci", "Cinema, sortides i subscripcions", False),
        ("Salut", "Farmàcia i metges", True)
    ]
    
    existing_cats = {c.name: c for c in session.query(Category).all()}
    for name, desc, essential in cat_data:
        if name not in existing_cats:
            session.add(Category(name=name, description=desc, is_essential=essential))

    # 2. MÈTODES DE PAGAMENT
    methods_data = [("Efectiu", "Cartera física"), ("Targeta Visa", "Banc Principal"), ("PayPal", "Pagaments online")]
    
    existing_methods = {m.name: m for m in session.query(PaymentMethod).all()}
    for name, provider in methods_data:
        if name not in existing_methods:
            session.add(PaymentMethod(name=name, provider=provider))

    # 3. TIPUS DE TRANSACCIÓ
    types_data = ["Ingrés", "Despesa"]
    existing_types = {t.name: t for t in session.query(TransactionType).all()}
    for name in types_data:
        if name not in existing_types:
            session.add(TransactionType(name=name))

    session.commit()
    print("Categories, mètodes i tipus carregats correctament.")

Categories, mètodes i tipus carregats correctament.


**Bloc 3: Usuari de prova i Perfil Financer**

In [3]:
with Session() as session:
    # Comprovem si l'usuari ja existeix
    user_email = "test@walletly.com"
    user = session.query(User).filter_by(email=user_email).first()

    if not user:
        # Creem usuari
        user = User(
            name="Joan",
            surname="Prova",
            email=user_email,
            password="hashed_password_123" # En una app real aniria encriptada
        )
        session.add(user)
        session.flush() # Necessitem l'ID de l'usuari abans del commit

        # Li creem el seu perfil financer
        profile = FinancialProfile(
            id_user=user.id_user,
            monthly_salary=Decimal("2000.00"),
            savings_goal=Decimal("400.00"),
            currency="EUR",
            risk_profile="Conservative"
        )
        session.add(profile)
        
    session.commit()
    print(f"Usuari {user_email} i perfil creats.")

Usuari test@walletly.com i perfil creats.


**Bloc 4: Pressupost inicial (Budget)**

In [4]:
with Session() as session:
    user = session.query(User).first()
    
    # Mirem si ja té un pressupost per a l'abril de 2026
    existing_budget = session.query(Budget).filter_by(
        id_user=user.id_user, month=4, year=2026
    ).first()

    if not existing_budget:
        budget = Budget(
            id_user=user.id_user,
            month=4,
            year=2026,
            description="Pressupost Abril inicial",
            total_limit=Decimal("1500.00")
        )
        session.add(budget)
        session.commit()
        print("Pressupost d'abril creat.")
    else:
        print("El pressupost ja existia.")

Pressupost d'abril creat.
